1. Instalar Essentia

In [ ]:
!pip install -q essentia-tensorflow

2. Subir canción

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
assert uploaded, "No se ha seleccionado ningún archivo"
audio_file = str(Path('/content') / next(iter(uploaded)))
print('Canción seleccionada:', audio_file)

Saving My Heart Will Go On  Celine Dion.mp3 to My Heart Will Go On  Celine Dion.mp3
Canción seleccionada: /content/My Heart Will Go On  Celine Dion.mp3


3. Descargar los dos modelos y sus etiquetas  
Essentia usa archivos .pb para los modelos TensorFlow y .json para la lista de etiquetas del clasificador

In [ ]:
import json
from pathlib import Path
from urllib.request import urlretrieve

model_dir = Path('/content/essentia_models')
model_dir.mkdir(parents=True, exist_ok=True)

urls = {
    'discogs-effnet-bs64-1.pb': (
        'https://essentia.upf.edu/models/feature-extractors/discogs-effnet/'
        'discogs-effnet-bs64-1.pb'
    ),
    'mtg_jamendo_top50tags-discogs-effnet-1.pb': (
        'https://essentia.upf.edu/models/classification-heads/mtg_jamendo_top50tags/'
        'mtg_jamendo_top50tags-discogs-effnet-1.pb'
    ),
    'mtg_jamendo_top50tags-discogs-effnet-1.json': (
        'https://essentia.upf.edu/models/classification-heads/mtg_jamendo_top50tags/'
        'mtg_jamendo_top50tags-discogs-effnet-1.json'
    ),
}

for filename, url in urls.items():
    destination = model_dir / filename
    if not destination.is_file():
        print('Descargando:', filename)
        urlretrieve(url, destination)
    else:
        print('Ya descargado:', filename)

print('Modelos disponibles en:', model_dir)

Descargando: discogs-effnet-bs64-1.pb
Descargando: mtg_jamendo_top50tags-discogs-effnet-1.pb
Descargando: mtg_jamendo_top50tags-discogs-effnet-1.json
Modelos disponibles en: /content/essentia_models


4. Obtener embeddings con Discogs-EffNet  
MonoLoader carga el MP3, lo convierte a mono y lo remustrea a 16kHz. Discogs-EffNet genera varios embeddings por canción, asociados a fragemntos temporales

In [ ]:
import numpy as np
from essentia.standard import MonoLoader, TensorflowPredictEffnetDiscogs, TensorflowPredict2D

audio = MonoLoader(filename=audio_file, sampleRate=16000, resampleQuality=4)()
print(f'Duración de audio analizada: {len(audio)/16000:.1f} segundos')

extractor = TensorflowPredictEffnetDiscogs(
    graphFilename=str(model_dir / 'discogs-effnet-bs64-1.pb'),
    output='PartitionedCall:1',
)
embeddings = np.asarray(extractor(audio))

print('Dimensiones de los embeddings:', embeddings.shape)
print('Primeros 8 valores del primer segmento:', embeddings[0, :8])

Duración de audio analizada: 283.1 segundos
Dimensiones de los embeddings: (284, 1280)
Primeros 8 valores del primer segmento: [ 0.00701504 -0.01269535  0.00668329 -0.00700603  0.20569594  0.04401869
  0.01579033  0.03192899]


5. Predecir etiquetas con MTG-Jamendo Top50 Tags  
El clasificador recibe los embeddings temporales. Agregamos con una media de las puntuaciones de cada etiqueta, no con una unión de embeddings. Las puntuaciones indican el grado de activación del modelo, no una certeza garantizada

In [ ]:
classifier = TensorflowPredict2D(
    graphFilename=str(model_dir / 'mtg_jamendo_top50tags-discogs-effnet-1.pb')
)
predictions = np.asarray(classifier(embeddings))

with open(model_dir / 'mtg_jamendo_top50tags-discogs-effnet-1.json', encoding='utf-8') as f:
    metadata = json.load(f)
labels = metadata['classes']

print('Dimensiones de predicciones:', predictions.shape)
print('Número de etiquetas del JSON:', len(labels))

scores = predictions.mean(axis=0)
if len(scores) != len(labels):
    raise ValueError(
        f'Incompatibilidad: el modelo devolvió {len(scores)} valores '
        f'pero el JSON declara {len(labels)} etiquetas. '
        'Comprueba que has descargado el par .pb/.json correcto.'
    )

import pandas as pd
results = pd.DataFrame({'etiqueta': labels, 'puntuacion': scores.astype(float)})
results = results.sort_values('puntuacion', ascending=False).reset_index(drop=True)

print('Las 15 etiquetas con mayor puntuación:')
display(results.head(15))

Dimensiones de predicciones: (284, 50)
Número de etiquetas del JSON: 50
Las 15 etiquetas con mayor puntuación:


,etiqueta,puntuacion
0,electronic,0.291887
1,pop,0.227134
2,synthesizer,0.200446
3,classical,0.163826
4,easylistening,0.151498
5,ambient,0.139172
6,soundtrack,0.135662
7,piano,0.128534
8,chillout,0.089756
9,rock,0.086842


6. Ver las 50 etiquetas y descargar resultados

In [ ]:
print('Todas las etiquetas, ordenadas:')
display(results)

csv_path = '/content/dj_copilot_etiquetas.csv'
results.to_csv(csv_path, index=False)
print('CSV guardado en:', csv_path)
files.download(csv_path)

Todas las etiquetas, ordenadas:


,etiqueta,puntuacion
0,electronic,0.291887
1,pop,0.227134
2,synthesizer,0.200446
3,classical,0.163826
4,easylistening,0.151498
5,ambient,0.139172
6,soundtrack,0.135662
7,piano,0.128534
8,chillout,0.089756
9,rock,0.086842


CSV guardado en: /content/dj_copilot_etiquetas.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>